<a href="https://www.kaggle.com/code/avikdas567/march-machine-learning-mania-2026?scriptVersionId=298972644" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2026/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv
/kaggle/input/march-machine-learning-mania

In [2]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import GroupKFold
from sklearn.metrics import brier_score_loss
import lightgbm as lgb

DATA_DIR = "/kaggle/input/march-machine-learning-mania-2026"


# Load data

def load_games(prefix):
    reg = pd.read_csv(f"{DATA_DIR}/{prefix}RegularSeasonCompactResults.csv")
    tour = pd.read_csv(f"{DATA_DIR}/{prefix}NCAATourneyCompactResults.csv")
    return pd.concat([reg, tour], ignore_index=True)

games = pd.concat([load_games("M"), load_games("W")], ignore_index=True)


# Build team-season features

stats = defaultdict(lambda: {"games": 0, "wins": 0, "pts_for": 0, "pts_against": 0})

for r in games.itertuples(index=False):
    stats[(r.Season, r.WTeamID)]["games"] += 1
    stats[(r.Season, r.WTeamID)]["wins"] += 1
    stats[(r.Season, r.WTeamID)]["pts_for"] += r.WScore
    stats[(r.Season, r.WTeamID)]["pts_against"] += r.LScore

    stats[(r.Season, r.LTeamID)]["games"] += 1
    stats[(r.Season, r.LTeamID)]["pts_for"] += r.LScore
    stats[(r.Season, r.LTeamID)]["pts_against"] += r.WScore

team_feats = pd.DataFrame([
    {
        "Season": s,
        "TeamID": t,
        "WinPct": v["wins"] / v["games"],
        "AvgMargin": (v["pts_for"] - v["pts_against"]) / v["games"],
        "Games": v["games"],
    }
    for (s, t), v in stats.items() if v["games"] > 0
])


# Build training matchups (OPTIMIZED LOOKUP)

team_feats_idx = team_feats.set_index(["Season", "TeamID"])

train_rows = []

for r in games.itertuples(index=False):
    t1, t2 = sorted([r.WTeamID, r.LTeamID])
    label = 1 if r.WTeamID == t1 else 0

    key1 = (r.Season, t1)
    key2 = (r.Season, t2)
    if key1 not in team_feats_idx.index or key2 not in team_feats_idx.index:
        continue

    f1 = team_feats_idx.loc[key1]
    f2 = team_feats_idx.loc[key2]

    train_rows.append({
        "Season": r.Season,
        "WinPctDiff": f1.WinPct - f2.WinPct,
        "MarginDiff": f1.AvgMargin - f2.AvgMargin,
        "GamesDiff": f1.Games - f2.Games,
        "Target": label
    })

train = pd.DataFrame(train_rows)

X = train[["WinPctDiff", "MarginDiff", "GamesDiff"]]
y = train["Target"]
groups = train["Season"]


# Cross-validation

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "verbosity": -1
}

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(train))

for tr, va in gkf.split(X, y, groups):
    model = lgb.train(
        params,
        lgb.Dataset(X.iloc[tr], y.iloc[tr]),
        num_boost_round=500
    )
    oof[va] = model.predict(X.iloc[va])

print("CV Brier score:", brier_score_loss(y, oof))


# Train final model

final_model = lgb.train(
    params,
    lgb.Dataset(X, y),
    num_boost_round=600
)


# Submission Generation

sample = pd.read_csv(f"{DATA_DIR}/SampleSubmissionStage2.csv")

ids = sample.ID.str.split("_", expand=True)
t1 = ids[1].astype(int)
t2 = ids[2].astype(int)

feat_2026 = team_feats[team_feats.Season == 2026].set_index("TeamID")

f1 = feat_2026.loc[t1].reset_index(drop=True)
f2 = feat_2026.loc[t2].reset_index(drop=True)

X_test = np.column_stack([
    f1.WinPct.values - f2.WinPct.values,
    f1.AvgMargin.values - f2.AvgMargin.values,
    f1.Games.values - f2.Games.values
])

sample["Pred"] = final_model.predict(X_test)
sample["Pred"] = sample["Pred"].clip(0.02, 0.98)

sample.to_csv("submission.csv", index=False)
print("submission.csv saved!")

CV Brier score: 0.1648992875603174
submission.csv saved!
